# kt-aivle-big-proj-vlm — Colab 서버 운영 가이드

**사전 준비**
1. 상단 메뉴 → 런타임 → 런타임 유형 변경 → **GPU (T4 이상)** 선택
2. 좌측 사이드바 🔑 아이콘 (Secrets) → `HF_TOKEN` 키로 HuggingFace 토큰 저장

셀을 위에서 아래로 순서대로 실행하세요.

| 엔드포인트 | 설명 |
|---|---|
| `GET  /health` | 헬스체크 |
| `POST /vlm/reports/daily` | 일일 보고서 생성 |
| `GET  /docs` | Swagger UI |

## 1단계: GPU 확인

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU를 찾을 수 없습니다. 런타임 유형을 GPU로 변경해 주세요.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

GPU: Tesla T4
VRAM: 15.6 GB
CUDA: 12.8


## 2단계: 레포 클론

In [16]:
import os

REPO_DIR = "/content/kt-aivle-big-proj-vlm"

if os.path.exists(REPO_DIR):
    print("이미 클론됨 — 최신화 중...")
    # test 브랜치로 가져오도록 지정
    !git -C {REPO_DIR} pull origin test
else:
    # clone 할 때 test 브랜치를 지정
    !git clone -b test https://github.com/aivle-bigproject-16/kt-aivle-big-proj-vlm.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"작업 디렉터리: {os.getcwd()}")

이미 클론됨 — 최신화 중...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 7 (delta 5), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 630 bytes | 630.00 KiB/s, done.
From https://github.com/aivle-bigproject-16/kt-aivle-big-proj-vlm
 * branch            test       -> FETCH_HEAD
   492ba2f..61e3da2  test       -> origin/test
Updating 492ba2f..61e3da2
Fast-forward
 app/schemas/request.py                | 5 ++---
 app/services/image_quality_service.py | 3 +--
 2 files changed, 3 insertions(+), 5 deletions(-)
작업 디렉터리: /content/kt-aivle-big-proj-vlm


## 3단계: 의존성 설치

> `transformers`를 git 최신 버전으로 설치하므로 3~5분 소요될 수 있습니다.

In [3]:
!pip install -q -r requirements.txt
print("✅ 설치 완료")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 115.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.5 MB/s eta 

## 4단계: HuggingFace 토큰 설정

좌측 🔑 Secrets 탭에 `HF_TOKEN`이 등록되어 있어야 합니다.

In [4]:
import os
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN 시크릿이 설정되지 않았습니다. 좌측 🔑 탭을 확인하세요.")

os.environ["HF_TOKEN"] = hf_token
print("✅ HF_TOKEN 설정 완료")

✅ HF_TOKEN 설정 완료


## 5단계: 모델 사전 다운로드 및 검증

`Qwen/Qwen3-VL-4B-Instruct` (~2.5 GB)를 HF 캐시에 내려받습니다.  
다운로드 진행 상황이 셀 출력에 실시간으로 표시됩니다.

In [5]:
import os

REPO_DIR = "/content/kt-aivle-big-proj-vlm"

if not os.path.exists(REPO_DIR):
    raise RuntimeError("레포 디렉터리가 없습니다. 2단계(클론) 셀을 먼저 실행하세요.")

%cd {REPO_DIR}
!python download_model.py

/content
GPU: Tesla T4
VRAM: 15.6 GB

✅ 저장된 토큰으로 자동 로그인

📦 모델 다운로드 시작: Qwen/Qwen3-VL-4B-Instruct
   (첫 실행 시 수 GB 다운로드 — 시간이 걸릴 수 있습니다)

config.json: 100% 1.50k/1.50k [00:00<00:00, 3.26MB/s]
model.safetensors.index.json: 100% 64.7k/64.7k [00:00<00:00, 136MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/4.97G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/8.88G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   4% 335M/8.88G [00:05<02:24, 58.9MB/s, 24.6MB/s  ]
Reconstructing (incomplete total...):  48% 4.29G/8.88G [00:59<00:58, 78.2MB/s, 43.4MB/s  ]
Reconstructing (incomplete total...):  82% 7.25G/8.88G [01:25<00:21, 76.4MB/s, 65.5MB/s  ]
Reconstructing (incomplete total...):  88% 7.83G/8.88G [01:25<00:10, 100MB/s, 84.2MB/s  ] 
Reconstructing (incomplete total...): 100% 8.88G/8.88G [01:29<00:00, 142MB/s, 72.0MB/s  ]

Fetching 2 fi

## 5단계: ngrok 설치

ngrok으로 외부 공개 URL을 생성합니다.  
[ngrok.com](https://ngrok.com) 무료 회원가입 후 **Auth Token**을 발급받아 Colab Secrets에 `NGROK_TOKEN` 키로 저장하세요.

In [6]:
!pip install -q pyngrok
print("✅ pyngrok 설치 완료")

✅ pyngrok 설치 완료


## 6단계: FastAPI 서버 시작 (포트 7860)

uvicorn을 백그라운드에서 실행합니다. 모델 로드까지 **5~10분** 소요됩니다.

In [17]:
import subprocess, time, requests, os, threading
import psutil, torch

REPO_DIR = "/content/kt-aivle-big-proj-vlm"

if not os.path.exists(REPO_DIR):
    raise RuntimeError("레포 디렉터리가 없습니다. 2단계(클론) 셀을 먼저 실행하세요.")

# 이전 uvicorn 프로세스만 골라서 종료 (Colab 내부 프로세스 보호)
result = subprocess.run(["pgrep", "-f", "uvicorn app.main:app"], capture_output=True, text=True)
for pid in result.stdout.strip().split("\n"):
    if pid.strip():
        subprocess.run(["kill", pid.strip()])
        print(f"이전 uvicorn 종료: PID {pid.strip()}")
time.sleep(1)

def mem_status():
    ram = psutil.virtual_memory()
    ram_used = ram.used / 1e9
    ram_total = ram.total / 1e9
    vram_used = torch.cuda.memory_allocated() / 1e9
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    return f"RAM {ram_used:.1f}/{ram_total:.1f}GB  |  VRAM {vram_used:.1f}/{vram_total:.1f}GB"

server_log = open("/content/server.log", "w")
server_proc = subprocess.Popen(
    [
        "python", "-m", "uvicorn", "app.main:app",
        "--host", "0.0.0.0",
        "--port", "7860",
        "--log-level", "info",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# 서버 로그를 콘솔 + 파일에 동시 출력하는 백그라운드 스레드
def _stream_log(proc, logfile):
    for line in proc.stdout:
        print(f"[server] {line}", end="", flush=True)
        logfile.write(line)
        logfile.flush()

log_thread = threading.Thread(target=_stream_log, args=(server_proc, server_log), daemon=True)
log_thread.start()

print(f"서버 PID: {server_proc.pid}")
print("모델 로드 대기 중...\n")
print(f"{'경과':>6}  {'상태':<10}  메모리")
print("-" * 50)

for i in range(120):  # 최대 10분
    time.sleep(5)
    elapsed = f"{(i+1)*5}초"
    try:
        r = requests.get("http://localhost:7860/health", timeout=2)
        if r.status_code == 200:
            print(f"{elapsed:>6}  {'✅ 준비완료':<10}  {mem_status()}")
            break
    except Exception:
        print(f"{elapsed:>6}  {'대기중':<10}  {mem_status()}")
else:
    print("\n⚠️  타임아웃")

이전 uvicorn 종료: PID 3256
[server] INFO:     Shutting down
[server] INFO:     Waiting for application shutdown.
[server] INFO:     Application shutdown complete.
[server] INFO:     Finished server process [3256]
서버 PID: 5415
모델 로드 대기 중...

    경과  상태          메모리
--------------------------------------------------
    5초  대기중         RAM 1.7/13.6GB  |  VRAM 0.0/15.6GB
   10초  대기중         RAM 1.9/13.6GB  |  VRAM 0.0/15.6GB
[server] INFO:     Started server process [5415]
[server] INFO:     Waiting for application startup.
[server] 
[server] Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 22919.69it/s]
[server] 
[server] Loading weights:   0%|          | 1/713 [00:02<32:14,  2.72s/it]
   15초  대기중         RAM 2.0/13.6GB  |  VRAM 0.0/15.6GB
[server] Loading weights:  10%|█         | 73/713 [00:07<00:47, 13.51it/s]
   20초  대기중         RAM 2.1/13.6GB  |  VRAM 0.0/15.6GB
[server] Loading weights:  21%|██        | 150/713 [00:12<00:29, 18.85it/s]
   25초  대기중         RAM 2.1/13.6GB  |  VRAM 0

## 7단계: 외부 공개 URL 생성 (ngrok 터널)

In [ ]:
ngrok.kill()

In [10]:
from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))

public_url = ngrok.connect(7860).public_url
print(f"🌐 공개 URL: {public_url}")
print(f"   헬스체크:    {public_url}/health")
print(f"   API 문서:    {public_url}/docs")
print(f"   보고서 생성: POST {public_url}/vlm/reports/daily")

🌐 공개 URL: https://grab-phony-conceded.ngrok-free.dev
   헬스체크:    https://grab-phony-conceded.ngrok-free.dev/health
   API 문서:    https://grab-phony-conceded.ngrok-free.dev/docs
   보고서 생성: POST https://grab-phony-conceded.ngrok-free.dev/vlm/reports/daily


---
## (선택) API 테스트

서버가 준비된 후 아래 셀로 보고서 생성을 테스트할 수 있습니다.

In [18]:
import requests, time
from IPython.display import display, Markdown

BASE_URL = "http://localhost:7860"  # ngrok 발급 URL을 사용할 경우 여기를 수정하세요

# 이미지 품질 검사 API 페이로드로 변경
payload = {
    "imageType": "RGB",
    "images": [
        {
            "imageId": "image1",
            "imageUrl": "./app/data/rgb_focus_failure.jpg"
        },
        {
            "imageId": "image2",
            "imageUrl": "./app/data/rgb_surface_dust.jpg"
        },
        {
            "imageId": "image3",
            "imageUrl": "./app/data/rgb_trigger_timing_failure.jpg"
        },
        {
            "imageId": "image4",
            "imageUrl": "./app/data/rgb_underexposure.jpg"
        },
        {
            "imageId": "image5",
            "imageUrl": "./app/data/rgb_hair_contamination.jpg"
        },
        {
            "imageId": "image6",
            "imageUrl": "./app/data/rgb_reflection_glare.jpg"
        },
        {
            "imageId": "image7",
            "imageUrl": "./app/data/rgb_uneven_lighting.jpg"
        },
        {
            "imageId": "image8",
            "imageUrl": "./app/data/rgb_NONE.jpg"
        },
        {
            "imageId": "image9",
            "imageUrl": "./app/data/rgb_overexposure.jpg"
        }
    ]
}

# 엔드포인트 변경
endpoint = f"{BASE_URL}/vlm/qualityInspection"

t0 = time.perf_counter()
print(f"요청 중: {endpoint}...")
resp = requests.post(endpoint, json=payload, timeout=600)
elapsed = time.perf_counter() - t0

try:
    resp.raise_for_status()
    data = resp.json()
    print(f"\n상태: {data.get('status')}")
    print(f"소요: {elapsed:.1f}초")

    # inspection_result가 반환된다고 가정
    print("\n결과 데이터:")
    print(data['content'])
except Exception as e:
    print(f"에러 발생: {e}")
    print(resp.text)

요청 중: http://localhost:7860/vlm/qualityInspection...
[server] 🔄 Qwen 추론 시작...
[server] 🔄 Qwen 추론 시작...
[server] 🔄 Qwen 추론 시작...
[server] 🔄 Qwen 추론 시작...
[server] 🔄 Qwen 추론 시작...
[server] INFO:     221.155.154.124:0 - "GET /docs HTTP/1.1" 200 OK
[server] INFO:     221.155.154.124:0 - "GET /openapi.json HTTP/1.1" 200 OK
[server] ✅ Qwen 추론 완료 — 71.5초
[server] ✅ Qwen 추론 완료 — 72.3초
[server] ✅ Qwen 추론 완료 — 75.5초
[server] ✅ Qwen 추론 완료 — 75.6초
[server] ✅ Qwen 추론 완료 — 76.3초

상태: COMPLETED
소요: 76.7초

결과 데이터:
[{'imageId': 'image1', 'failType': 'rgb_focus_failure', 'description': '[OpenCV 검출] 선명도 점수(1.1) 미달로 초점 불량.'}, {'imageId': 'image3', 'failType': 'rgb_focus_failure', 'description': '[OpenCV 검출] 선명도 점수(4.8) 미달로 초점 불량.'}, {'imageId': 'image4', 'failType': 'rgb_underexposure', 'description': '[OpenCV 검출] 평균 밝기(65.5) 미달로 노출 부족.'}, {'imageId': 'image9', 'failType': 'rgb_overexposure', 'description': '[OpenCV 검출] 평균 밝기(199.0) 초과로 노출 과다.'}, {'imageId': 'image2', 'failType': 'rgb_surface_dust', 'desc

In [19]:
import requests, time
from IPython.display import display, Markdown

BASE_URL = "http://localhost:7860"  # ngrok 발급 URL을 사용할 경우 여기를 수정하세요

# 이미지 품질 검사 API 페이로드로 변경
payload = {
    "imageType": "CT",
    "images": [
        {
            "imageId": "image1",
            "imageUrl": "./app/data/ct_acquisition_motion.jpg"
        },
        {
            "imageId": "image2",
            "imageUrl": "./app/data/ct_beam_hardening_metal_streak.jpg"
        },
        {
            "imageId": "image3",
            "imageUrl": "./app/data/ct_cell_alignment_failure.jpg"
        },
        {
            "imageId": "image4",
            "imageUrl": "./app/data/ct_insufficient_projection_sampling.jpg"
        },
        {
            "imageId": "image5",
            "imageUrl": "./app/data/ct_low_signal_noise.jpg"
        },
        {
            "imageId": "image6",
            "imageUrl": "./app/data/ct_NONE.jpg"
        }
    ]
}

# 엔드포인트 변경
endpoint = f"{BASE_URL}/vlm/qualityInspection"

t0 = time.perf_counter()
print(f"요청 중: {endpoint}...")
resp = requests.post(endpoint, json=payload, timeout=600)
elapsed = time.perf_counter() - t0

try:
    resp.raise_for_status()
    data = resp.json()
    print(f"\n상태: {data.get('status')}")
    print(f"소요: {elapsed:.1f}초")

    # inspection_result가 반환된다고 가정
    print("\n결과 데이터:")
    print(data['content'])
except Exception as e:
    print(f"에러 발생: {e}")
    print(resp.text)

요청 중: http://localhost:7860/vlm/qualityInspection...
[server] 🔄 Qwen 추론 시작...
[server] 🔄 Qwen 추론 시작...
[server] 🔄 Qwen 추론 시작...
[server] 🔄 Qwen 추론 시작...
[server] ✅ Qwen 추론 완료 — 41.5초
[server] ✅ Qwen 추론 완료 — 43.5초
[server] ✅ Qwen 추론 완료 — 44.7초
[server] ✅ Qwen 추론 완료 — 46.2초
[server] INFO:     127.0.0.1:44086 - "POST /vlm/qualityInspection HTTP/1.1" 200 OK

상태: COMPLETED
소요: 46.3초

결과 데이터:
[{'imageId': 'image3', 'failType': 'ct_cell_alignment_failure', 'description': '[OpenCV 검출] 배터리 좌우가 화면 경계에 닿음. (X 시작: 2, 끝: 113)'}, {'imageId': 'image5', 'failType': 'ct_low_signal_noise', 'description': '[OpenCV 검출] 평균 노이즈 수치(4.7) 초과로 화질 저하.'}, {'imageId': 'image1', 'failType': 'ct_insufficient_projection_sampling', 'description': '외곽선이 계단처럼 깨져 보이며, 구조 주변으로 얕은 줄무늬(Streak)가 전체적으로 발생함.'}, {'imageId': 'image2', 'failType': 'ct_insufficient_projection_sampling', 'description': '외곽선이 계단처럼 깨져 보이며, 구조 주변으로 얕은 줄무늬(Streak)가 전체적으로 발생함.'}, {'imageId': 'image4', 'failType': 'ct_beam_hardening_metal_streak', 'descr

## (선택) 서버 종료

In [ ]:
from pyngrok import ngrok

server_proc.terminate()
ngrok.kill()
print("서버 및 터널 종료됨")